<a href="https://colab.research.google.com/github/ameemaiqbal/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ameemaiqbal/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: Score each page by combining two signals from my ML-05 feature vector: how much its ranking position bounces around (position_volatility) and how much its impressions dropped between the two 15-day halves. A page scores higher (more urgent to review) when it has BOTH high volatility AND a real impression drop, since Notebook 03/ML-02 found volatility alone predicts decline better than impressions alone, but a page with both is the clearest case.

Reason codes the rule can output:

HIGH_VOLATILITY_AND_DROP: both signals present, top priority
HIGH_VOLATILITY_ONLY: unstable ranking but impressions haven't dropped yet, early warning candidate
IMPRESSION_DROP_ONLY: impressions fell but ranking is stable, may be a demand/seasonality issue rather than a ranking issue
NEITHER: no concerning signal, lowest priority

In [6]:
print("Rule: score = normalized(position_volatility) + normalized(impression_drop_pct)")
print("Reason codes: HIGH_VOLATILITY_AND_DROP, HIGH_VOLATILITY_ONLY, IMPRESSION_DROP_ONLY, NEITHER")


Rule: score = normalized(position_volatility) + normalized(impression_drop_pct)
Reason codes: HIGH_VOLATILITY_AND_DROP, HIGH_VOLATILITY_ONLY, IMPRESSION_DROP_ONLY, NEITHER


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
%pip -q install duckdb huggingface_hub
import os, getpass, pandas as pd
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
}

features = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily_sample']}),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last15,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev15,
               STDDEV(CASE WHEN f.report_date > b.end_d - INTERVAL 15 DAY THEN f.gsc_avg_position END) AS position_volatility
        FROM {TABLES['fact_daily_sample']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 30 DAY
        GROUP BY 1, 2
        HAVING imp_prev15 >= 50
    )
    SELECT * FROM windowed
""").df()

content_meta = con.sql(f"SELECT client_hash_id, content_hash_id, content_type FROM {TABLES['dim_content']}").df()
data = features.merge(content_meta, on=['client_hash_id', 'content_hash_id'], how='left')
data['content_type'] = data['content_type'].fillna('unknown')
data = data.dropna(subset=['position_volatility'])
print(f"{len(data):,} rows ready")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

99,181 rows ready


In [8]:
import numpy as np
import os

data['impression_drop_pct'] = (data['imp_prev15'] - data['imp_last15']) / data['imp_prev15'].clip(lower=1)

def normalize(s):
    return (s - s.min()) / (s.max() - s.min())

data['volatility_norm'] = normalize(data['position_volatility'])
data['drop_norm'] = normalize(data['impression_drop_pct'].clip(lower=0))

data['action_score'] = data['volatility_norm'] + data['drop_norm']

high_vol = data['volatility_norm'] > 0.5
has_drop = data['impression_drop_pct'] > 0.2

conditions = [
    high_vol & has_drop,
    high_vol & ~has_drop,
    ~high_vol & has_drop,
]
choices = ['HIGH_VOLATILITY_AND_DROP', 'HIGH_VOLATILITY_ONLY', 'IMPRESSION_DROP_ONLY']
data['reason_code'] = np.select(conditions, choices, default='NEITHER')

queue = data.sort_values('action_score', ascending=False)

os.makedirs('work/outputs', exist_ok=True)
queue[['client_hash_id', 'content_hash_id', 'content_type', 'position_volatility',
       'impression_drop_pct', 'action_score', 'reason_code']].to_csv(
    'work/outputs/baseline_action_score.csv', index=False)

print(f"Queue written: {len(queue):,} rows")
print(queue['reason_code'].value_counts())
queue.head(10)

Queue written: 99,181 rows
reason_code
IMPRESSION_DROP_ONLY        49683
NEITHER                     49454
HIGH_VOLATILITY_AND_DROP       32
HIGH_VOLATILITY_ONLY           12
Name: count, dtype: int64


,client_hash_id,content_hash_id,imp_last15,imp_prev15,position_volatility,content_type,impression_drop_pct,volatility_norm,drop_norm,action_score,reason_code
33651,client_23a62021009f63c4,content_b152c38d97ca782a,5.0,64.0,150.884821,keyword article,0.921875,0.866535,0.922364,1.788899,HIGH_VOLATILITY_AND_DROP
53246,client_20259bd6705d81d4,content_4689bd28a6c49125,17.0,76.0,174.124237,keyword article,0.776316,1.000000,0.776728,1.776728,HIGH_VOLATILITY_AND_DROP
35721,client_23a62021009f63c4,content_9ccdeb0b66a4e7f3,17.0,85.0,164.747847,keyword article,0.800000,0.946151,0.800424,1.746576,HIGH_VOLATILITY_AND_DROP
84583,client_23a62021009f63c4,content_ac7a84ddc8b50684,14.0,343.0,131.974038,keyword article,0.959184,0.757930,0.959693,1.717623,HIGH_VOLATILITY_AND_DROP
79938,client_62f4a7e64f5e0096,content_0a7eab9b100980e4,4.0,128.0,129.424689,keyword article,0.968750,0.743289,0.969264,1.712553,HIGH_VOLATILITY_AND_DROP
86064,client_23a62021009f63c4,content_7309c1ba02a6a9be,72.0,369.0,146.913807,keyword article,0.804878,0.843730,0.805305,1.649035,HIGH_VOLATILITY_AND_DROP
29781,client_62f4a7e64f5e0096,content_446d68987f97953a,4.0,51.0,125.641023,keyword article,0.921569,0.721560,0.922058,1.643617,HIGH_VOLATILITY_AND_DROP
51410,client_62f4a7e64f5e0096,content_3cf49a9a7e001c60,2.0,67.0,115.258405,keyword article,0.970149,0.661932,0.970664,1.632596,HIGH_VOLATILITY_AND_DROP
70156,client_62f4a7e64f5e0096,content_3163b02b3578e188,5.0,109.0,111.799821,keyword article,0.954128,0.642069,0.954635,1.596704,HIGH_VOLATILITY_AND_DROP
68856,client_0fa64a184f18a4a0,content_b35ee8fea4f7b163,5.0,50.0,114.737381,keyword article,0.900000,0.658940,0.900477,1.559417,HIGH_VOLATILITY_AND_DROP


In [9]:
top20 = queue.head(20)[['content_hash_id', 'content_type', 'imp_last15', 'imp_prev15',
                          'position_volatility', 'impression_drop_pct', 'action_score', 'reason_code']]
top20

,content_hash_id,content_type,imp_last15,imp_prev15,position_volatility,impression_drop_pct,action_score,reason_code
33651,content_b152c38d97ca782a,keyword article,5.0,64.0,150.884821,0.921875,1.788899,HIGH_VOLATILITY_AND_DROP
53246,content_4689bd28a6c49125,keyword article,17.0,76.0,174.124237,0.776316,1.776728,HIGH_VOLATILITY_AND_DROP
35721,content_9ccdeb0b66a4e7f3,keyword article,17.0,85.0,164.747847,0.800000,1.746576,HIGH_VOLATILITY_AND_DROP
84583,content_ac7a84ddc8b50684,keyword article,14.0,343.0,131.974038,0.959184,1.717623,HIGH_VOLATILITY_AND_DROP
79938,content_0a7eab9b100980e4,keyword article,4.0,128.0,129.424689,0.968750,1.712553,HIGH_VOLATILITY_AND_DROP
86064,content_7309c1ba02a6a9be,keyword article,72.0,369.0,146.913807,0.804878,1.649035,HIGH_VOLATILITY_AND_DROP
29781,content_446d68987f97953a,keyword article,4.0,51.0,125.641023,0.921569,1.643617,HIGH_VOLATILITY_AND_DROP
51410,content_3cf49a9a7e001c60,keyword article,2.0,67.0,115.258405,0.970149,1.632596,HIGH_VOLATILITY_AND_DROP
70156,content_3163b02b3578e188,keyword article,5.0,109.0,111.799821,0.954128,1.596704,HIGH_VOLATILITY_AND_DROP
68856,content_b35ee8fea4f7b163,keyword article,5.0,50.0,114.737381,0.900000,1.559417,HIGH_VOLATILITY_AND_DROP


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

#	content_hash_id (short)	Action	Confidence note	What would make it wrong
1-10, 12-15	b152c3, 4689bd, 9ccdeb, ac7a84, 0a7eab, 7309c1, 446d68, 3cf49a, 3163b0, b35ee8, 193c4f, 5a150f, fe1492, d890df	Flag for manual review	Low confidence — imp_prev15 mostly under 100 (some as low as 50-67), so position_volatility is computed from very few data points, could be noise, not a real trend	Wrong if the low base volume means the volatility number is statistically unstable rather than a genuine pattern; would need more days of history to confirm
11, 16-19	eed097, d2a80e, 79ee6f, 72f880, 2c597e	Flag for manual review	Low-moderate confidence — similarly low volume (50-190 impressions), same small-sample caveat applies	Same as above — needs a longer observation window to separate signal from noise
20	e37858	Priority review, higher confidence	Higher confidence — imp_prev15 = 6,848, a real volume base, this decline (93.6% drop) is measured on solid data, not noise	Wrong only if something external (a known site-wide event, deliberate de-indexing, or a tracking/tagging issue) explains the drop, not the content itself

Overall pattern: 19 of the top 20 are all keyword article type with genuinely low impression volume (mostly under 200), meaning the score is currently dominated by statistically noisy volatility rather than confirmed, high-confidence decline. Only row 20 (e37858, 6,848 impressions) is a high-confidence pick.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: 19 of the top 20 (all except e378585d3c36b91e) have low base impression volume (imp_prev15 mostly 50-200). Since position_volatility is a standard deviation calculated over only 15 days of already-sparse traffic, these high volatility numbers likely reflect statistical noise from small sample sizes rather than genuine ranking instability. This is a real weakness in the current rule: it doesn't account for confidence/sample size, a page with 3 impressions bouncing around looks identical to a genuinely volatile high-traffic page in this score. A fix for a future iteration would be to require a minimum volume threshold (e.g. imp_prev15 >= 200) before trusting the volatility signal, or to weight the score by log(volume) to naturally downweight noisy small-sample cases.

Leakage check: confirmed no product decision flags or future-window data leaked into this score. The two features used, position_volatility and impression_drop_pct, are both derived purely from fact_daily_sample's gsc_avg_position and gsc_impressions, columns that reflect observed historical search performance, not internal product flags. Critically, last_optimized_date and optimization_eligible_date, confirmed as leaky in ML-05 (12.3% of dated rows fell after the prediction window), were not used anywhere in this scoring rule, verified by checking the code above: only position_volatility, impression_drop_pct, and content_type (for context) appear in the action_score calculation.

In [10]:
score_inputs = ['position_volatility', 'impression_drop_pct']
leaky_fields = ['last_optimized_date', 'optimization_eligible_date']
print("Fields used in action_score calculation:", score_inputs)
print("Confirmed leaky fields NOT among them:", all(f not in score_inputs for f in leaky_fields))


Fields used in action_score calculation: ['position_volatility', 'impression_drop_pct']
Confirmed leaky fields NOT among them: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.